# Python reference 9: dates, times and time zones

*Author: [Sai Guntupalli](https://github.com/sai-guntupalli) · [sai-guntupalli/data-engineering-handbook](https://github.com/sai-guntupalli/data-engineering-handbook) · CC BY 4.0*

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/sai-guntupalli/data-engineering-handbook/blob/master/notebooks/topics/python/learning/python-reference-09-dates-times-and-time-zones.ipynb)

`date`, `datetime` and `timedelta` arithmetic; naive versus aware datetimes, `zoneinfo` and daylight-saving gaps and folds; ISO 8601, `strftime`/`strptime` and Unix timestamps; month arithmetic, date ranges and truncation; and which clock to use for timing. Practise it in [Python practice 9: dates, times and time zones](../practice/python-practice-09-dates-times-and-time-zones.ipynb).

*Last reviewed: November 2026, against Python 3.13.*

## Contents

| section | entries |
| --- | --- |
| [The datetime types](#the-datetime-types) | [`date`, `time`, `datetime`, `timedelta`](#date-time-datetime-timedelta) |
| [Time zones](#time-zones) | [Naive and aware datetimes](#naive-and-aware-datetimes), [Daylight saving time](#daylight-saving-time) |
| [Parsing and formatting](#parsing-and-formatting) | [ISO 8601, `strftime` and `strptime`](#iso-8601-strftime-and-strptime), [Unix timestamps](#unix-timestamps) |
| [Calendar arithmetic](#calendar-arithmetic) | [Month ends, adding months and date ranges](#month-ends-adding-months-and-date-ranges), [Measuring elapsed time](#measuring-elapsed-time) |

## The datetime types

### `date`, `time`, `datetime`, `timedelta`

| type | holds | construct |
| --- | --- | --- |
| `date` | year, month, day | `date(2026, 11, 27)`, `date.today()`, `date.fromisoformat("2026-11-27")` |
| `time` | hour … microsecond, optional tzinfo | `time(9, 30)` |
| `datetime` | a date and a time, optional tzinfo | `datetime(2026, 11, 27, 9, 30)`, `datetime.now(timezone.utc)`, `datetime.combine(d, t)` |
| `timedelta` | a duration: days, seconds, microseconds | `timedelta(days=1, hours=2)`; `.total_seconds()` |

| operation | result |
| --- | --- |
| `datetime ± timedelta` | `datetime` |
| `datetime - datetime` | `timedelta` (both naive or both aware) |
| `dt.date()`, `dt.time()` | parts |
| `dt.replace(hour=0, minute=0, second=0, microsecond=0)` | a changed copy (truncation) |
| `d.weekday()` / `d.isoweekday()` / `d.isocalendar()` | Mon=0 / Mon=1 / `(iso_year, week, weekday)` |

**Notes:** `timedelta` has no months or years, because their length varies; month arithmetic needs `calendar`
(below) or `dateutil.relativedelta`. `timedelta` normalises to days, seconds and microseconds, so
`timedelta(hours=36).days` is 1 and `.seconds` is 43200 (not the total); use `total_seconds()`.

In [ ]:
from datetime import date, datetime, time, timedelta, timezone

d = date(2026, 11, 27)
dt = datetime(2026, 11, 27, 9, 30)
print(d + timedelta(days=5), dt - timedelta(hours=10), datetime(2026, 12, 25) - dt)
print(d.weekday(), d.isoweekday(), tuple(d.isocalendar()), dt.replace(minute=0), datetime.combine(d, time(23, 59)))
span = timedelta(hours=36)
print(span.days, span.seconds, span.total_seconds(), timedelta(minutes=90) // timedelta(minutes=15))

## Time zones

### Naive and aware datetimes

A **naive** datetime has `tzinfo=None`: it is a wall-clock reading with no zone, and Python cannot convert or
reliably compare it. An **aware** datetime carries a `tzinfo` and denotes an exact instant.

| task | code |
| --- | --- |
| current instant | `datetime.now(timezone.utc)` (never `datetime.utcnow()`: naive and deprecated) |
| attach a zone to a wall-clock time | `naive.replace(tzinfo=ZoneInfo("Europe/London"))` |
| convert an instant to another zone | `aware.astimezone(ZoneInfo("Asia/Kolkata"))` |
| to UTC | `aware.astimezone(timezone.utc)` |
| fixed offset | `timezone(timedelta(hours=5, minutes=30))` |

**Notes:** store and compute in UTC; convert to local time only for display and for local business rules ("orders
before 17:00 local ship today"). Comparing naive with aware raises `TypeError`, and subtracting them does too.

In [ ]:
from zoneinfo import ZoneInfo

utc_now = datetime(2026, 11, 27, 14, 0, tzinfo=timezone.utc)
print(utc_now.astimezone(ZoneInfo("America/New_York")), utc_now.astimezone(ZoneInfo("Asia/Kolkata")))
local = datetime(2026, 11, 27, 9, 0).replace(tzinfo=ZoneInfo("Europe/London"))
print(local.astimezone(timezone.utc), local.utcoffset(), local.tzname())
try:
    datetime(2026, 1, 1) < utc_now
except TypeError as e:
    print("TypeError:", e)

### Daylight saving time

`zoneinfo.ZoneInfo` (3.9+, the IANA database) applies daylight-saving rules. Twice a year local time is irregular:

| case | example (America/New_York) | Python behaviour |
| --- | --- | --- |
| **gap** (spring forward) | 02:30 on 2026-03-08 does not exist | the datetime is created anyway; converting to UTC normalises it |
| **fold** (fall back) | 01:30 on 2026-11-01 happens twice | `fold=0` is the first (EDT), `fold=1` the second (EST) |

**Notes:** arithmetic on aware datetimes in the same `ZoneInfo` is **wall-clock** arithmetic: adding
`timedelta(hours=24)` across a DST change keeps the local hour, so the elapsed time is 23 or 25 hours. For elapsed
time, convert to UTC, add, and convert back. Local days are not always 24 hours long.

In [ ]:
ny = ZoneInfo("America/New_York")
gap = datetime(2026, 3, 8, 2, 30, tzinfo=ny)
print("gap:", gap, "->", gap.astimezone(timezone.utc))
first, second = datetime(2026, 11, 1, 1, 30, tzinfo=ny), datetime(2026, 11, 1, 1, 30, fold=1, tzinfo=ny)
print("fold:", first.astimezone(timezone.utc), second.astimezone(timezone.utc))

start = datetime(2026, 3, 7, 12, 0, tzinfo=ny)
wall = start + timedelta(hours=24)
elapsed = (start.astimezone(timezone.utc) + timedelta(hours=24)).astimezone(ny)
print("wall-clock +24h:", wall, "| elapsed +24h:", elapsed)
day_start, next_day = datetime(2026, 3, 8, tzinfo=ny), datetime(2026, 3, 9, tzinfo=ny)
print("hours in 2026-03-08 local:", (next_day.astimezone(timezone.utc) - day_start.astimezone(timezone.utc)) / timedelta(hours=1))

## Parsing and formatting

### ISO 8601, `strftime` and `strptime`

| call | does |
| --- | --- |
| `dt.isoformat(sep="T", timespec="auto")` | `2026-11-27T09:30:00+00:00`; `timespec="seconds"` / `"milliseconds"` |
| `datetime.fromisoformat(s)` | parses ISO strings, including `Z` and offsets (3.11+) |
| `dt.strftime(fmt)` | format with directives |
| `datetime.strptime(s, fmt)` | parse with directives; `ValueError` on mismatch |

| directive | meaning | directive | meaning |
| --- | --- | --- | --- |
| `%Y` / `%y` | 2026 / 26 | `%H` / `%I` `%p` | 24-hour / 12-hour with AM/PM |
| `%m` / `%d` | zero-padded month / day | `%M` / `%S` / `%f` | minute / second / microsecond |
| `%b` / `%B` | Nov / November (locale-dependent) | `%z` / `%Z` | `+0000` offset / zone name |
| `%a` / `%A` | Fri / Friday | `%j` / `%G-W%V` | day of year / ISO week |

**Notes:** prefer ISO 8601 with an offset for anything machine-readable. `strptime` with `%z` returns an aware
datetime; without it the result is naive. Formatting is also available in f-strings: `f"{dt:%Y-%m-%d}"`.

In [ ]:
dt = datetime(2026, 11, 27, 9, 30, 15, 123456, tzinfo=timezone.utc)
print(dt.isoformat(), dt.isoformat(timespec="milliseconds"), f"{dt:%Y-%m-%d %H:%M}", dt.strftime("%a %d %b %Y, %I:%M %p"))
print(datetime.fromisoformat("2026-11-27T09:30:00Z"), datetime.fromisoformat("2026-11-27 09:30+05:30").utcoffset())
print(datetime.strptime("27/11/2026 09:30", "%d/%m/%Y %H:%M"), datetime.strptime("2026-11-27 09:30 +0100", "%Y-%m-%d %H:%M %z"))
try:
    datetime.strptime("2026-13-01", "%Y-%m-%d")
except ValueError as e:
    print("ValueError:", e)

### Unix timestamps

| call | does |
| --- | --- |
| `dt.timestamp()` | seconds since 1970-01-01 UTC as a float (a naive `dt` is treated as **local** time) |
| `datetime.fromtimestamp(ts, tz=timezone.utc)` | an aware UTC datetime; without `tz` you get naive local time |
| `ts_ms / 1000` | many systems (Kafka, JavaScript, Spark `long` columns) use milliseconds |

In [ ]:
ts = datetime(2026, 11, 27, tzinfo=timezone.utc).timestamp()
print(ts, datetime.fromtimestamp(ts, tz=timezone.utc), datetime.fromtimestamp(1_795_737_600_123 / 1000, tz=timezone.utc))

## Calendar arithmetic

### Month ends, adding months and date ranges

| task | approach |
| --- | --- |
| days in a month | `calendar.monthrange(year, month)[1]` |
| first day of the month | `d.replace(day=1)` |
| last day of the month | `d.replace(day=monthrange(d.year, d.month)[1])` |
| add months | compute year and month by integer arithmetic, then clamp the day to the month's length |
| a range of dates | a generator adding `timedelta(days=1)` |
| truncate to an interval | `dt - (dt - epoch) % interval` for fixed intervals (15 minutes, 1 hour) |

In [ ]:
import calendar

def add_months(d, months):
    index = d.month - 1 + months
    year, month = d.year + index // 12, index % 12 + 1
    return d.replace(year=year, month=month, day=min(d.day, calendar.monthrange(year, month)[1]))

def date_range(start, end, step=timedelta(days=1)):
    current = start
    while current <= end:
        yield current
        current += step

def floor_to(dt, interval):
    epoch = datetime(1970, 1, 1, tzinfo=dt.tzinfo)
    return dt - (dt - epoch) % interval

print(calendar.monthrange(2028, 2)[1], add_months(date(2026, 1, 31), 1), add_months(date(2026, 11, 30), 3), add_months(date(2026, 3, 15), -14))
print([str(x) for x in date_range(date(2026, 12, 30), date(2027, 1, 2))])
print(floor_to(datetime(2026, 11, 27, 9, 47, 12, tzinfo=timezone.utc), timedelta(minutes=15)))

### Measuring elapsed time

| clock | use |
| --- | --- |
| `time.perf_counter()` | durations in code (highest resolution, monotonic) |
| `time.monotonic()` | timeouts and deadlines (never jumps backwards) |
| `time.time()` / `datetime.now(timezone.utc)` | wall-clock timestamps for logs and records (can jump when the clock is adjusted) |

In [ ]:
import time

start = time.perf_counter()
sum(range(1_000_000))
print(f"took {time.perf_counter() - start:.4f}s")

**Next:** practise these entries in [Python practice 9: dates, times and time zones](../practice/python-practice-09-dates-times-and-time-zones.ipynb).